# Attention — NLP → LLM Notes

## Learning Path

```
Tokenization → Token IDs → Embeddings → Word2Vec → RNN → LSTM
→ ATTENTION  ← (current topic)
→ Self-Attention → Query / Key / Value (Q/K/V) → Scaled Dot-Product Attention
→ Causal Attention / Masking → Multi-Head Attention → Transformer → GPT → Modern GPT / LLM
```

## Table of Contents
1. What is Attention?
2. Why Did We Need Attention?
3. Basic Attention Idea
4. Softmax
5. Attention Weights
6. Weighted Sum
7. Basic Attention Formula
8. The Important Idea
9. Query, Key, Value (Q/K/V)
10. Attention Flow
11. Why Three Things?
12. Self-Attention
13. Example of Self-Attention
14. Basic Q/K/V Mathematics
15. Scaled Dot-Product Attention
16. Dot Product
17. Attention Matrix
18. Softmax is Applied Row-wise
19. Causal Attention
20. Causal Mask
21. Attention vs Self-Attention
22. Attention vs RNN/LSTM
23. Why Attention Was Important
24. Attention Is Not Yet a Transformer
25. Multi-Head Attention
26. Attention Dimensions
27. Important Shape Flow
28. PyTorch Connection
29. Your ModernGPT Connection
30. Embedding vs Attention
31. Word2Vec vs RNN vs LSTM vs Attention
32. Simple Mental Model
33. Most Important Formula
34. One-Line Summary
35. Next Topic

---
# Part 1 — Foundations
## 1. What is Attention?

Attention is a mechanism that allows a neural network to decide **which parts of the input are important** when producing an output.

> **Simple idea:** "Look at all the available information, but focus more on the important parts."

**Example**

Sentence: *"The boy who was playing near the river picked up the ball."*

When the model later processes a pronoun like **"he"**, it should understand that "he" is related to **"boy"**. Attention helps the model give higher importance to relevant words instead of depending only on one hidden state.

## 2. Why Did We Need Attention?

Before attention, we learned **RNN** and **LSTM**.

**RNN** — information is passed sequentially:

```
word1 → hidden1 → word2 → hidden2 → word3 → hidden3 → word4 → hidden4
```

**Problem:** with a very long sequence, important information that is far away is difficult to remember or use directly.

**LSTM** improves this using:
- Forget gate
- Input gate
- Output gate
- Cell state

But LSTM is **still sequential**.

**Attention provides a different idea:**

```
Current token → Look at other tokens → Decide which are important → Collect useful information
```

## 3. Basic Attention Idea

Suppose we have four token embeddings: `I`, `love`, `Python`, `programming`.

We want the model to decide how important each token is. Example importance scores:

| Token | Score |
|---|---|
| I | 1 |
| love | 4 |
| Python | 2 |
| programming | 3 |

Higher score = more relevant. But raw scores are not convenient probabilities, so we apply **softmax**.

## 4. Softmax

Softmax converts scores into values between 0 and 1, and all values add up to 1.

Example: `scores = [1, 4, 2, 3]` → softmax ≈ `[0.032, 0.644, 0.087, 0.237]`

| Token | Share |
|---|---|
| I | 3.2% |
| love | **64.4%** |
| Python | 8.7% |
| programming | 23.7% |

`love` receives the highest attention.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)   # numerical stability
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

scores = np.array([1, 4, 2, 3], dtype=float)
weights = softmax(scores)
print("Softmax :", np.round(weights, 3))
print("Sum     :", weights.sum())

## 5. Attention Weights

The softmax values are called **attention weights**. They tell us how much importance the model gives to each piece of information.

| Token | Attention weight |
|---|---|
| I | 0.032 |
| love | 0.644 |
| Python | 0.087 |
| programming | 0.237 |

The weights are then used to combine information.

## 6. Weighted Sum

Suppose the value vectors are:

| Token | Value vector |
|---|---|
| I | [1, 0] |
| love | [0, 2] |
| Python | [2, 1] |
| programming | [1, 3] |

Attention weights: `[0.032, 0.644, 0.087, 0.237]`

```
output = 0.032 × I + 0.644 × love + 0.087 × Python + 0.237 × programming
       ≈ [0.443, 2.086]
```

This is the **attention output**.

In [ ]:
values = np.array([[1, 0],    # I
                   [0, 2],    # love
                   [2, 1],    # Python
                   [1, 3]],   # programming
                  dtype=float)

output = weights @ values      # Σ(weight × value)
print("Attention output:", np.round(output, 3))

## 7. Basic Attention Formula

```
Attention Output = Σ (weight × value)
```

In other words: **attention output = weighted combination of information**.

## 8. The Important Idea

Attention does **NOT** simply average all tokens. It gives *different* weights to different tokens.

```
Without attention:   A + B + C + D           → equal importance
With attention:      A×0.10  B×0.60  C×0.20  D×0.10  → more importance to B
```

---
# Part 2 — Query, Key, Value & Self-Attention
## 9. Query, Key, Value — Q / K / V

Modern attention uses three representations:

| Symbol | Name | Question it answers |
|---|---|---|
| **Q** | Query | *What am I looking for?* |
| **K** | Key | *What information do I contain / what am I relevant for?* |
| **V** | Value | *What information should I provide?* |

**Database analogy**

- Query: "I want information about Python."
- Keys: "Python programming", "Django", "Football", "Cooking"
- The Query is compared with the Keys → the most relevant Keys receive higher scores → their Values are collected.

## 10. Attention Flow

```
QUERY → Compare with Keys → Attention Scores → Softmax → Attention Weights → Multiply Values → Weighted Sum / Output
```

Short form:

```
Q × K  →  Scores  →  Softmax  →  Weights  →  Weights × V  →  Output
```

## 11. Why Three Things?

Query, Key and Value have different jobs:

- **Query:** "What am I looking for?"
- **Key:** "How can I be matched against the query?"
- **Value:** "If you select me, what information should I give?"

This separation makes attention flexible.

## 12. Self-Attention

When the Query, Key and Value all come from the **same sequence**, it is called **self-attention**.

Example: *"I love Python"* — every token produces its own Q, K and V:

```
I      → Q, K, V
love   → Q, K, V
Python → Q, K, V
```

Then each token can look at the other tokens.

## 13. Example of Self-Attention

Sentence: *"I love Python"*. For the token **"Python"**:

1. Python creates a Query.
2. That Query is compared with I's Key, love's Key and Python's Key.

Example scores:

| Pair | Score |
|---|---|
| Python → I | 1.0 |
| Python → love | 2.5 |
| Python → Python | 4.0 |

Softmax converts these into attention weights (≈ 0.04, 0.18, 0.79 — computed below). The Values are then combined using those weights.

In [ ]:
s = np.array([1.0, 2.5, 4.0])
print("Python's attention weights [I, love, Python]:", np.round(softmax(s), 2))

## 14. Basic Q/K/V Mathematics

Let Q = Query matrix, K = Key matrix, V = Value matrix.

1. `Q × Kᵀ` → similarity / relevance scores
2. `Softmax(Q × Kᵀ)` → attention weights
3. `Softmax(Q × Kᵀ) × V` → attention output

---
# Part 3 — Scaled Dot-Product Attention
## 15. Scaled Dot-Product Attention

Modern Transformers use:

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^{T}}{\sqrt{d_k}}\right)V$$

where **Q** = Query, **K** = Key, **V** = Value, **dₖ** = dimension of the Key vectors.

**Why divide by √dₖ?** When vector dimensions become large, `QKᵀ` values can become very large, which makes softmax too extreme. Scaling keeps values more stable.

## 16. Dot Product

The dot product measures how strongly two vectors align.

```
A = [1, 2],  B = [3, 4]
A · B = (1×3) + (2×4) = 3 + 8 = 11
```

Attention uses this to compare **Query ↔ Key**. Higher score = more relevant; lower score = less relevant.

In [ ]:
A = np.array([1, 2]); B = np.array([3, 4])
print("A · B =", np.dot(A, B))

## 17. Attention Matrix

For 3 tokens (`I`, `love`, `Python`), each token creates a Query, and each Query compares with every Key → a **3 × 3 attention score matrix**.

|  | Key: I | Key: love | Key: Python |
|---|---|---|---|
| **Query I** | ? | ? | ? |
| **Query love** | ? | ? | ? |
| **Query Python** | ? | ? | ? |

Example matrix:

```
[[2.0, 1.0, 0.5],
 [1.0, 3.0, 2.0],
 [0.5, 2.0, 4.0]]
```

- Each **row** belongs to one Query.
- Each **column** belongs to one Key.

## 18. Softmax Is Applied Row-wise

Each Query needs its own attention distribution, so softmax is applied across the **Key dimension** (each row).

Query "Python": `[0.5, 2.0, 4.0]` → softmax → `[small, medium, large]` (sums to 1).

In [ ]:
score_matrix = np.array([[2.0, 1.0, 0.5],
                         [1.0, 3.0, 2.0],
                         [0.5, 2.0, 4.0]])

attn = softmax(score_matrix, axis=-1)   # row-wise
print(np.round(attn, 3))
print("Row sums:", attn.sum(axis=1))

---
# Part 4 — Causal Attention (GPT)
## 19. Causal Attention

GPT is an **autoregressive** language model. When predicting the next token, it must **NOT** look into the future.

Example: *"I love Python programming"*. When predicting at **Python**, the model can see `I`, `love` but must NOT see `programming`.

Therefore GPT uses a **causal mask**.

## 20. Causal Mask

|  | I | love | Python |
|---|:-:|:-:|:-:|
| **I** | ✓ | ✗ | ✗ |
| **love** | ✓ | ✓ | ✗ |
| **Python** | ✓ | ✓ | ✓ |

✓ = allowed, ✗ = blocked. Mathematically, blocked positions receive **−∞** before softmax:

```
[2.0, -inf, -inf]  →  softmax  →  [1.0, 0.0, 0.0]
```

So a token cannot attend to future tokens.

In [ ]:
T = score_matrix.shape[0]
mask = np.triu(np.ones((T, T), dtype=bool), k=1)      # True = future (blocked)
masked_scores = np.where(mask, -np.inf, score_matrix)
causal_attn = softmax(masked_scores, axis=-1)
print(np.round(causal_attn, 3))

## 21. Attention vs Self-Attention

- **Attention:** general mechanism of selecting important information.
- **Self-attention:** Queries, Keys and Values come from the *same* sequence.
- **GPT uses:** self-attention — specifically **causal self-attention**.

## 22. Attention vs RNN / LSTM

```
RNN:        Token 1 → Token 2 → Token 3 → Token 4      (sequential processing)
LSTM:       Token 1 → Token 2 → Token 3 → Token 4      (better memory using gates)
Attention:  Token 1 ┐
            Token 2 ┼──→ Attention    (tokens can directly interact)
            Token 3 ┤
            Token 4 ┘
```

## 23. Why Attention Was Important

Attention made it much easier for models to connect distant parts of a sequence.

*"The cat that was sitting on the table near the window suddenly jumped because it heard a noise."*

A model may need to connect **"it"** with **"cat"**. Attention allows direct interaction between token representations.

---
# Part 5 — Towards Transformers
## 24. Attention Is Not Yet a Transformer

**Attention ≠ Transformer.** Attention is one major component of a Transformer.

Simplified Transformer block:

```
Input → Self-Attention → Add & Norm → Feed Forward Network → Add & Norm → Output
```

Modern GPT adds many additional details.

## 25. Multi-Head Attention

One attention mechanism can learn one type of relationship. Transformers instead use multiple attention **heads**:

- Head 1 → subject–verb relationship
- Head 2 → relationship between nearby words
- Head 3 → semantic relationship
- Head 4 → another learned relationship

Each head learns its own Q/K/V projections, and then the outputs are combined.

## 26. Attention Dimensions

Let **B** = batch size, **T** = sequence length, **E** = embedding dim. Input shape: `[B, T, E]`.

Example from your ModernGPT: `B = 16`, `T = 128`, `E = 256` → input `[16, 128, 256]`.

With `NUM_HEADS = 4`: `HEAD_DIM = 256 / 4 = 64`, so **4 heads × 64 dims = 256 total dims**.

In [ ]:
B, T, E, NUM_HEADS = 16, 128, 256, 4
HEAD_DIM = E // NUM_HEADS
print("Input shape :", (B, T, E))
print("HEAD_DIM    :", HEAD_DIM)
print("Check       :", NUM_HEADS, "×", HEAD_DIM, "=", NUM_HEADS * HEAD_DIM)

## 27. Important Shape Flow

```
Token IDs                    [B, T]
  ↓ Embedding
                             [B, T, E]
  ↓ Q/K/V projections
Q, K, V                      [B, T, E]   each
  ↓ split into heads
                             [B, Heads, T, HeadDim]
  ↓ Q × Kᵀ
                             [B, Heads, T, T]
  ↓ Softmax                  → attention weights
  ↓ weights × V
                             [B, Heads, T, HeadDim]
  ↓ merge heads
                             [B, T, E]
```

In [ ]:
# Trace the shape flow with random tensors (numpy)
rng = np.random.default_rng(0)
x = rng.standard_normal((B, T, E))
Wq, Wk, Wv = (rng.standard_normal((E, E)) * 0.02 for _ in range(3))

q, k, v = x @ Wq, x @ Wk, x @ Wv
print("Q/K/V          :", q.shape)

split = lambda t: t.reshape(B, T, NUM_HEADS, HEAD_DIM).transpose(0, 2, 1, 3)
qh, kh, vh = split(q), split(k), split(v)
print("Split heads    :", qh.shape)

scores = qh @ kh.transpose(0, 1, 3, 2) / np.sqrt(HEAD_DIM)
print("Q·Kᵀ / √d      :", scores.shape)

scores = np.where(np.triu(np.ones((T, T), dtype=bool), 1), -np.inf, scores)   # causal
w = softmax(scores, axis=-1)
out = w @ vh
print("weights × V    :", out.shape)

merged = out.transpose(0, 2, 1, 3).reshape(B, T, E)
print("Merged heads   :", merged.shape)

## 28. PyTorch Connection

Modern PyTorch provides `F.scaled_dot_product_attention()`:

```python
output = F.scaled_dot_product_attention(q, k, v, is_causal=True)
```

In your ModernGPT code, this is the actual attention calculation. `is_causal=True` means: **do not allow a token to attend to future tokens.**

In [ ]:
# Requires PyTorch: pip install torch
import torch
import torch.nn.functional as F

q_t = torch.randn(B, NUM_HEADS, T, HEAD_DIM)
k_t = torch.randn(B, NUM_HEADS, T, HEAD_DIM)
v_t = torch.randn(B, NUM_HEADS, T, HEAD_DIM)

out_t = F.scaled_dot_product_attention(q_t, k_t, v_t, is_causal=True)
print(out_t.shape)   # [B, Heads, T, HeadDim]

## 29. Your ModernGPT Connection

Your ModernGPT contains: **RMSNorm, RoPE, Q/K/V projections, Scaled Dot-Product Attention, Causal Attention, Multi-Head Attention, SwiGLU**.

The attention part follows this general flow:

```
Token Embeddings → Q / K / V → RoPE → Q × Kᵀ / √d → Causal Mask → Softmax
→ Attention Weights → × V → Attention Output
```

---
# Part 6 — Summary & Review
## 30. Embedding vs Attention (Very Important)

- **Embedding** answers: *"What is this token represented as?"* — e.g. `Python → [0.2, -0.7, 0.5, ...]`
- **Attention** answers: *"Which other tokens should this token pay attention to?"*

Example — how much should **Python** use information from:

| Token | Weight |
|---|---|
| I | 0.1 |
| love | 0.3 |
| Python | 0.6 |

## 31. Word2Vec vs RNN vs LSTM vs Attention

| Model | Idea | Notes |
|---|---|---|
| **Word2Vec** | Context → Target | Learns word embeddings |
| **RNN** | Sequence → Hidden State → Prediction | Processes sequentially |
| **LSTM** | Sequence → Gates + Cell State → Prediction | Better long-term memory than basic RNN |
| **Attention** | Query → compare with Keys → weighted Values | Directly connects relevant information |
| **Transformer** | Self-Attention + FFN + Residual Connections + Normalization | |
| **GPT** | Transformer decoder-style + Causal Self-Attention + Next-Token Prediction | |

## 32. Simple Mental Model

- **RNN:** "I will carry information forward."
- **LSTM:** "I will decide what information to remember and forget."
- **Attention:** "I will look at the relevant information directly."
- **Transformer:** "I will use attention + neural processing to understand relationships between tokens."
- **GPT:** "I will use a Transformer to predict the next token."

## 33. Most Important Formula to Remember

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^{T}}{\sqrt{d_k}}\right)V$$

1. Compare Query with Keys
2. Scale the scores
3. Softmax → attention weights
4. Use weights on Values
5. Get attention output

## 34. One-Line Summary

- **Attention:** allows each token to decide which other pieces of information are important and combine them using learned weights.
- **Self-attention:** every token can look at other tokens in the same sequence.
- **GPT:** uses causal self-attention so a token cannot see future tokens while predicting the next token.

In [ ]:
def scaled_dot_product_attention(Q, K, V, causal=False):
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)
    if causal:
        T = scores.shape[-1]
        scores = np.where(np.triu(np.ones((T, T), dtype=bool), 1), -np.inf, scores)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

# Quick test: 3 tokens, 4-dim vectors
rng = np.random.default_rng(1)
Q = K = V = rng.standard_normal((3, 4))
out, w = scaled_dot_product_attention(Q, K, V, causal=True)
print("Weights:\n", np.round(w, 3))
print("Output shape:", out.shape)

## 35. Next Topic — Implement Attention Ourselves

Learning order:

1. Simple attention with vectors
2. Dot-product attention
3. Q / K / V in PyTorch
4. Scaled dot-product attention
5. Causal mask
6. Multi-head attention
7. Transformer block
8. GPT
9. Your ModernGPT code

---
*End of Attention Notes*